In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install gradio praat-parselmouth librosa shap -q
print("✅ 完成")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 60.8 MB/s eta 0:00:00
✅ 完成


In [2]:
import pickle
import numpy as np
import librosa
import parselmouth
from parselmouth.praat import call
import shap

BASE_DIR = '/content/drive/MyDrive/voice_age_project_0428/voice_age_project_0428'

with open(f'{BASE_DIR}/model_artifacts.pkl', 'rb') as f:
    artifacts = pickle.load(f)

xgb_model = artifacts['xgb_model']
scaler    = artifacts['scaler']
le        = artifacts['label_encoder']
feature_cols = artifacts['feature_cols']

print("✅ 模型載入完成")
print(f"類別：{le.classes_}")

✅ 模型載入完成
類別：['Middle (36-50)' 'Senior (51+)' 'Young (20-35)']


In [3]:
def extract_features(wav_path):
    try:
        sound = parselmouth.Sound(wav_path)

        pitch = call(sound, "To Pitch", 0.0, 75, 600)
        f0_mean = call(pitch, "Get mean", 0, 0, "Hertz")

        point_process = call(sound, "To PointProcess (periodic, cc)", 75, 600)
        jitter = call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
        shimmer = call([sound, point_process],
                       "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)

        harmonicity = call(sound, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
        hnr = call(harmonicity, "Get mean", 0, 0)

        y, sr = librosa.load(wav_path, sr=16000)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_means = np.mean(mfccs, axis=1)

        features = {
            'F0_mean': f0_mean  if not np.isnan(f0_mean)  else 0.0,
            'Jitter':  jitter   if not np.isnan(jitter)   else 0.0,
            'Shimmer': shimmer  if not np.isnan(shimmer)  else 0.0,
            'HNR':     hnr      if not np.isnan(hnr)      else 0.0,
        }
        for i, val in enumerate(mfcc_means):
            features[f'MFCC_{i+1}'] = val
        return features
    except:
        return None

In [4]:
def predict_age(wav_path):
    feats = extract_features(wav_path)
    if feats is None:
        return None, None, None

    X = np.array([[feats[col] for col in feature_cols]])
    X_scaled = scaler.transform(X)

    # 預測類別與機率
    pred_idx = xgb_model.predict(X_scaled)[0]
    pred_proba = xgb_model.predict_proba(X_scaled)[0]
    pred_label = le.inverse_transform([pred_idx])[0]

    # SHAP（針對這一筆）
    explainer = shap.TreeExplainer(xgb_model)
    shap_vals = explainer.shap_values(X_scaled)  # (1, 17, 3)

    return pred_label, pred_proba, shap_vals, X_scaled, feats

In [5]:
import gradio as gr
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# 顏色對應
GROUP_COLOR = {
    'Young (20-35)':  '#4CAF50',
    'Middle (36-50)': '#2196F3',
    'Senior (51+)':   '#FF5722'
}

GROUP_EMOJI = {
    'Young (20-35)':  '🟢',
    'Middle (36-50)': '🔵',
    'Senior (51+)':   '🔴'
}

def analyze_voice(audio_path):
    if audio_path is None:
        return "請先錄音或上傳音檔", None, None

    result = predict_age(audio_path)
    if result[0] is None:
        return "❌ 音檔分析失敗，請重試", None, None

    pred_label, pred_proba, shap_vals, X_scaled, feats = result

    # ── 文字結果 ──────────────────────────────
    emoji = GROUP_EMOJI[pred_label]
    color = GROUP_COLOR[pred_label]

    prob_str = "\n".join([
        f"  {GROUP_EMOJI[le.classes_[i]]} {le.classes_[i]}: {p*100:.1f}%"
        for i, p in enumerate(pred_proba)
    ])

    result_text = f"""
{emoji} Predicted Age Group: {pred_label}

Confidence:
{prob_str}

── Vocal Biomarkers ──
  F0 (mean pitch):  {feats['F0_mean']:.1f} Hz
  Jitter:           {feats['Jitter']*100:.3f} %
  Shimmer:          {feats['Shimmer']*100:.3f} %
  HNR:              {feats['HNR']:.1f} dB
"""

    # ── 機率長條圖 ────────────────────────────
    fig1, ax1 = plt.subplots(figsize=(6, 3))
    colors = [GROUP_COLOR[c] for c in le.classes_]
    bars = ax1.barh(le.classes_, pred_proba * 100, color=colors)
    ax1.set_xlabel('Probability (%)')
    ax1.set_title('Age Group Prediction Probability')
    ax1.set_xlim(0, 100)
    for bar, p in zip(bars, pred_proba):
        ax1.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{p*100:.1f}%', va='center', fontsize=11)
    plt.tight_layout()

    # ── SHAP 條形圖（這一筆的解釋）─────────────
    senior_idx = list(le.classes_).index('Senior (51+)')
    shap_for_senior = shap_vals[0, :, senior_idx]  # shape: (17,)

    sorted_idx = np.argsort(np.abs(shap_for_senior))[-10:]  # 前10大

    fig2, ax2 = plt.subplots(figsize=(6, 4))
    bar_colors = ['#FF5722' if v > 0 else '#2196F3'
                  for v in shap_for_senior[sorted_idx]]
    ax2.barh(
        [feature_cols[i] for i in sorted_idx],
        shap_for_senior[sorted_idx],
        color=bar_colors
    )
    ax2.axvline(0, color='black', linewidth=0.8)
    ax2.set_xlabel('SHAP value (→ pushes toward Senior)')
    ax2.set_title('Why did AI make this prediction?')
    plt.tight_layout()

    return result_text, fig1, fig2


# ── 介面設計 ──────────────────────────────────
demo = gr.Interface(
    fn=analyze_voice,
    inputs=gr.Audio(
        sources=["microphone", "upload"],
        type="filepath",
        label="🎤 Record or Upload Your Voice (3 seconds)"
    ),
    outputs=[
        gr.Textbox(label="📊 Prediction Result", lines=12),
        gr.Plot(label="🎯 Probability Distribution"),
        gr.Plot(label="🔍 SHAP Explanation (Why this prediction?)")
    ],
    title="🎙️ AI Vocal Age Analyzer",
    description="""
**Vocal Biomarker & Age Detection System**
Speak into the microphone for 3 seconds. The AI will analyze your vocal features and predict your age group.

*Features analyzed: F0 (pitch), Jitter, Shimmer, HNR, and 13-dimensional MFCC*
""",
    examples=None,
    theme=gr.themes.Soft()
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d1a02829194780743b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
